# Test standardized personas

In [1]:
%load_ext autoreload
%autoreload 2

import importlib
import sys
from pathlib import Path

import pandas as pd

CODE_DIR = Path("../..").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from data import persona_loaders, persona_spec
importlib.reload(persona_spec)
importlib.reload(persona_loaders)

specs = persona_spec.load_persona_spec()
combined = persona_loaders.load_all_personas(specs=specs)

print(f"Loaded {len(combined):,} participants | active attributes: {list(specs)}")
for persona in specs.keys():
    print(persona)
    print(combined[persona].unique())
    print('+'*50)

combined.head()

Loaded 49,468 participants | active attributes: ['country', 'gender', 'age', 'region', 'state', 'education', 'family_income', 'income', 'race', 'hispanic', 'religion', 'religious_attendance', 'political_party', 'political_ideology', 'marital_status', 'employment_status', 'household_size', 'children_in_household', 'children_ever_had', 'urban_rural', 'voted_last_election', 'union_member', 'residence_type', 'citizen', 'employer_sector', 'spouse_education', 'spouse_employment', 'nutrition_label_use', 'ancestry', 'personal_income', 'occupation', 'town_size', 'immigrant_status', 'lives_with_parents', 'home_language', 'chief_wage_earner', 'family_savings', 'social_class', 'voting_frequency']
country
['United States' 'Canada']
++++++++++++++++++++++++++++++++++++++++++++++++++
gender
['Male' 'Female' nan 'Prefer not to say' 'Non-binary' 'Other']
++++++++++++++++++++++++++++++++++++++++++++++++++
age
['65+' '55-64' '18-24' '25-34' '35-44' '45-54' nan '18-29' '50-64' '30-49'
 '65-74' '75-84' '85

,dataset,CASEID,country,gender,age,region,state,education,family_income,income,...,personal_income,occupation,town_size,immigrant_status,lives_with_parents,home_language,chief_wage_earner,family_savings,social_class,voting_frequency
0,issp,2015:2015008400000003,United States,Male,65+,New England,NaN,"Lower level tertiary, first stage","$75,000-$89,999",NaN,...,No income,Information and communications technology inst...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,issp,2015:2015008400000005,United States,Female,55-64,New England,NaN,"Upper level tertiary (Master, Doctor)","$170,000 or more",NaN,...,"$1,000-$2,999",Shop sales assistants,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,issp,2015:2015008400000008,United States,Female,18-24,Middle Atlantic,NaN,Upper secondary education,"$30,000-$34,999",NaN,...,"$15,000-$17,499",Home-based personal care workers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,issp,2015:2015008400000010,United States,Male,65+,Middle Atlantic,NaN,Post-secondary non-tertiary education,"$60,000-$74,999",NaN,...,No income,Heavy truck and lorry drivers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,issp,2015:2015008400000011,United States,Female,25-34,Middle Atlantic,NaN,Upper secondary education,"Less than $1,000",NaN,...,No income,Health care assistants,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
combined['political_ideology'].value_counts(dropna=False)

political_ideology
NaN                          37403
Moderate                      2203
Liberal                        985
Very liberal                   927
5 (0 = Left, 10 = Right)       847
Conservative                   765
Very conservative              760
5 (1 = Left, 10 = Right)       615
Somewhat conservative          548
Somewhat liberal               446
3 (1 = Left, 10 = Right)       308
10 (0 = Left, 10 = Right)      268
6 (1 = Left, 10 = Right)       260
1 (1 = Left, 10 = Right)       248
7 (1 = Left, 10 = Right)       242
7 (0 = Left, 10 = Right)       239
8 (1 = Left, 10 = Right)       239
6 (0 = Left, 10 = Right)       206
4 (1 = Left, 10 = Right)       206
3 (0 = Left, 10 = Right)       204
4 (0 = Left, 10 = Right)       203
8 (0 = Left, 10 = Right)       182
10 (1 = Left, 10 = Right)      182
2 (0 = Left, 10 = Right)       164
0 (0 = Left, 10 = Right)       145
2 (1 = Left, 10 = Right)       135
Slightly liberal               130
Slightly conservative          127
9

In [2]:
# dataset_order = list(persona_loaders.RAW_LOADERS.keys())
# for attr in specs:
#     present_ds = [ds for ds in dataset_order if combined.loc[combined["dataset"] == ds, attr].notna().any()]
#     sub = combined.loc[combined["dataset"].isin(present_ds), ["dataset", attr]].copy()
#     sub[attr] = sub[attr].where(sub[attr].notna(), "<NA>").astype(str)
#     print(f"\n=== {attr} (non-null: {int((sub[attr] != '<NA>').sum()):,}, <NA>: {int((sub[attr] == '<NA>').sum()):,}, unique: {combined[attr].nunique()}) ===")
#     table = pd.crosstab(sub[attr], sub["dataset"], margins=True)[present_ds + ["All"]]
#     display(table)

## Per-dataset persona value counts

In [3]:
def inspect_dataset_personas(ds_name: str) -> None:
    sub = combined[combined["dataset"] == ds_name]
    if sub.empty:
        raise ValueError(f"Unknown or empty dataset {ds_name!r}")
    active_attrs = [a for a in specs if a in sub.columns and sub[a].notna().any()]
    n = len(sub)
    print(f"=== {ds_name}: {n:,} participants | {len(active_attrs)} active persona attributes ===")
    for attr in active_attrs:
        n_present = int(sub[attr].notna().sum())
        n_missing = n - n_present
        n_unique = int(sub[attr].nunique(dropna=True))
        s = sub[attr].where(sub[attr].notna(), "<NA>").astype(str)
        vc = s.value_counts()
        tbl = pd.DataFrame({"count": vc, "pct": (vc / n * 100).round(2)})
        tbl.index.name = attr
        print(f"\n{attr} (present={n_present:,}, missing={n_missing:,}, unique={n_unique})")
        display(tbl)

# ISSP

In [4]:
inspect_dataset_personas("issp")

=== issp: 14,365 participants | 22 active persona attributes ===

country (present=14,365, missing=0, unique=1)


,count,pct
country,,
United States,14365,100.0



gender (present=14,335, missing=30, unique=2)


,count,pct
gender,,
Female,7906,55.04
Male,6429,44.75
<NA>,30,0.21



age (present=13,949, missing=416, unique=6)


,count,pct
age,,
65+,3522,24.52
55-64,2500,17.40
35-44,2441,16.99
25-34,2374,16.53
45-54,2141,14.90
18-24,971,6.76
<NA>,416,2.90


... (34 outputs left out)


personal_income (present=13,502, missing=863, unique=27)


,count,pct
personal_income,,
No income,5088,35.42
<NA>,863,6.01
"$60,000-$74,999",848,5.90
"$40,000-$49,999",760,5.29
"$50,000-$59,999",718,5.00
"$75,000-$89,999",570,3.97
"$30,000-$34,999",529,3.68
"$90,000-$109,999",483,3.36
"$35,000-$39,999",462,3.22



occupation (present=13,702, missing=663, unique=322)


,count,pct
occupation,,
<NA>,663,4.62
Primary school teachers,359,2.50
Business services and administration managers not elsewhere classified,332,2.31
Nursing professionals,326,2.27
Manufacturing labourers not elsewhere classified,315,2.19
...,...,...
Enquiry clerks,1,0.01
Hunters and trappers,1,0.01
Other cleaning workers,1,0.01


# Twin2k

In [5]:
inspect_dataset_personas("twin2k")

=== twin2k: 2,058 participants | 15 active persona attributes ===

country (present=2,058, missing=0, unique=1)


,count,pct
country,,
United States,2058,100.0



gender (present=2,058, missing=0, unique=2)


,count,pct
gender,,
Female,1044,50.73
Male,1014,49.27



age (present=2,058, missing=0, unique=4)


,count,pct
age,,
30-49,735,35.71
50-64,658,31.97
18-29,388,18.85
65+,277,13.46


... (20 outputs left out)


household_size (present=2,058, missing=0, unique=5)


,count,pct
household_size,,
2,650,31.58
3,423,20.55
1,412,20.02
4,352,17.10
More than 4,221,10.74



citizen (present=2,058, missing=0, unique=2)


,count,pct
citizen,,
Yes,2054,99.81
No,4,0.19


# WVS7

In [6]:
inspect_dataset_personas("wvs7")

=== wvs7: 2,596 participants | 29 active persona attributes ===

country (present=2,596, missing=0, unique=1)


,count,pct
country,,
United States,2596,100.0



gender (present=2,596, missing=0, unique=2)


,count,pct
gender,,
Male,1390,53.54
Female,1206,46.46



age (present=2,596, missing=0, unique=6)


,count,pct
age,,
25-34,702,27.04
35-44,455,17.53
45-54,433,16.68
55-64,379,14.60
65+,340,13.10
18-24,287,11.06


... (48 outputs left out)


social_class (present=2,561, missing=35, unique=5)


,count,pct
social_class,,
Lower middle class,967,37.25
Upper middle class,762,29.35
Working class,615,23.69
Lower class,183,7.05
<NA>,35,1.35
Upper class,34,1.31



voting_frequency (present=2,562, missing=34, unique=4)


,count,pct
voting_frequency,,
Always,1582,60.94
Usually,546,21.03
Never,356,13.71
Not allowed to vote,78,3.00
<NA>,34,1.31


# Subpop

In [7]:
inspect_dataset_personas("subpop")

=== subpop: 22 participants | 9 active persona attributes ===

country (present=22, missing=0, unique=1)


,count,pct
country,,
United States,22,100.0



gender (present=2, missing=20, unique=2)


,count,pct
gender,,
<NA>,20,90.91
Male,1,4.55
Female,1,4.55



region (present=2, missing=20, unique=2)


,count,pct
region,,
<NA>,20,90.91
Northeast,1,4.55
South,1,4.55


... (8 outputs left out)


political_party (present=2, missing=20, unique=2)


,count,pct
political_party,,
<NA>,20,90.91
Democrat,1,4.55
Republican,1,4.55



political_ideology (present=3, missing=19, unique=3)


,count,pct
political_ideology,,
<NA>,19,86.36
Liberal,1,4.55
Conservative,1,4.55
Moderate,1,4.55


# Covid

In [ ]:
inspect_dataset_personas("covid")

# Grummon 2023

In [ ]:
inspect_dataset_personas("grummon2023")

# Grummon 2025

In [ ]:
inspect_dataset_personas("grummon2025")

# Berke 2023

In [ ]:
inspect_dataset_personas("berke2023")

# Kriner 2026

In [ ]:
inspect_dataset_personas("kriner2026")

# Wu 2025

In [ ]:
inspect_dataset_personas("wu2025")

# Kantorowicz 2026

In [ ]:
inspect_dataset_personas("kantorowicz2026")

# Magistro 2025

In [ ]:
inspect_dataset_personas("magistro2025")